#### 문항 1. 12개월간 유지율 하락을 경영진이 30초 안에 이해하는 추세 차트 만들기

**상황**

구독 서비스의 3개월 유지율이 1년 사이 계속 떨어지고 있습니다. 다음 주 경영 회의에서 대표이사에게 보고할 첫 장을 만들어야 합니다.  

청중: 대표이사  

청중의 질문: **"지금 무슨 일이 벌어지고 있는가?"**  

차트의 목적: **추세** — 시간에 따라 어떻게 변했는가

<br>

**요구사항**

7개 항목을 모두 만족해야 합니다.  
<br>  

**R1.** 차트 종류

12개 시점의 방향이 메시지이므로 선 그래프로 그린다. 막대는 구간이 많아지면 추세가 묻힌다

**R2.** 비교 기준선

연초 값(71%)을 가로 점선으로 긋고 라벨을 단다. 기준선이 없으면 "54%가 나쁜 건가?" 라는 질문이 반드시 나온다

**R3.** 낙폭 강조

기준선과 실제 선 사이를 옅은 색으로 채워 하락 폭을 면적으로 보여준다

**R4.** 직접 라벨링

마지막 시점의 값을 점 옆에 직접 표기한다. 범례 사용 금지

**R5.** 차트 정크 제거

위·오른쪽·아래 축선 제거, y축 눈금은 3개 이하로

**R6.** Action Title

제목을 결론 문장으로. 부제로 무슨 데이터인지 한 줄

**R7.** 메타 정보

하단에 출처 · 기간 · N수 · 단위 캡션

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager


# ══════════════════════════════════════════════════════════════════════
# 0. 환경 설정 — 수정하지 마세요
# ══════════════════════════════════════════════════════════════════════
INK, BODY, MUTED = '#0c0a09', '#4e4e4e', '#777169'
HAIR, GRAY, GRAY_D = '#d6d3d1', '#c9c5c1', '#a8a29e'
ACCENT, WARN = '#2f6f5e', '#c2543d'

def setup_font():
    '''설치된 한글 폰트를 자동으로 찾아 적용한다.'''
    candidates = [
        'Malgun Gothic',       # Windows
        'AppleGothic',         # macOS
    ]
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            plt.rcParams['font.family'] = name
            break
    else:
        print('[경고] 한글 폰트를 찾지 못했습니다. 라벨이 □로 보일 수 있습니다.\n'
              '       Colab: !apt-get install -y fonts-nanum 후 런타임 재시작')
    plt.rcParams.update({
        'axes.unicode_minus': False,   # 음수 부호 깨짐 방지
    })

def save(fig, path):
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)


# ══════════════════════════════════════════════════════════════════════
# 1. 데이터 — 수정하지 마세요
# ══════════════════════════════════════════════════════════════════════
def load_data():
    '''가입 월별 신규 가입자 수와 3개월 유지율 (2025년 12개월)
    '''
    return pd.DataFrame({
        '가입월': pd.date_range('2025-01-01', periods=12, freq='MS'),
        '3개월유지율': [71.0, 70.0, 69.0, 68.0, 66.0, 64.0,
                        61.0, 59.0, 57.0, 56.0, 55.0, 54.0],
        '신규가입자': [8200, 8600, 9100, 9400, 10200, 11800,
                       13500, 14100, 13800, 13200, 12600, 12100],
    })

SOURCE_NOTE = ('자료: 구독 관리 DB · 기간 2025.01~2025.12 · '
               'N=신규 가입 137,600명 · 단위 %')

# ══════════════════════════════════════════════════════════════════════
# 2. 여기부터 작성하세요
# ══════════════════════════════════════════════════════════════════════
def my_answer():
    df = load_data()
    x = df['가입월']
    y = df['3개월유지율']

    fig, ax = plt.subplots(figsize=(8.6, 4.7))

    # ── R1. 선 그래프 ─────────────────────────────────────────────
    ax.plot(x, y, linewidth=2.2, color=WARN, zorder=3)

    # ── R2. 기준선 ────────────────────────────────────────────────
    start_val = y.iloc[0]  # 71.0 (1월 값)
    ax.axhline(start_val, color=MUTED, linestyle='--', linewidth=1, zorder=1)
    ax.text(x.iloc[0], start_val + 1.2, f'연초 {start_val:.0f}%',
            fontsize=9, color=MUTED)

    # ── R3. 낙폭 음영 ─────────────────────────────────────────────
    ax.fill_between(x, y, start_val, color=WARN, alpha=0.12, zorder=2)

    # ── R4. 마지막 값 직접 라벨링 ─────────────────────────────────
    last_x, last_y = x.iloc[-1], y.iloc[-1]
    ax.scatter([last_x], [last_y], color=WARN, s=45, zorder=4)
    ax.text(last_x, last_y - 3, f'{last_y:.0f}%',
            fontsize=13, color=WARN, fontweight='bold', ha='center')

    drop = start_val - last_y  # 17
    ax.annotate('', xy=(last_x, last_y + 1.5), xytext=(last_x, start_val - 1),
                arrowprops=dict(arrowstyle='<->', color=INK, lw=1))
    ax.text(last_x + pd.Timedelta(days=12), (start_val + last_y) / 2,
            f'-{drop:.0f}%p', fontsize=10, color=INK, va='center')

    # ── R5. 차트 정크 제거 ────────────────────────────────────────
    for side in ['top', 'right', 'bottom']:
        ax.spines[side].set_visible(False)
    ax.set_yticks([50, 60, 70])
    ax.set_ylim(48, 75)

    # x축 라벨 설정 (참고용)
    ax.set_xticks(x[::2], [t.strftime('%y-%m') for t in x[::2]], fontsize=9)
    ax.tick_params(length=0, colors=MUTED)


    # ── R6. Action Title ──────────────────────────────────────────
    ax.set_title('3개월 유지율이 1년 만에 71% → 54%로 17%p 하락했다',
                  loc='left', fontsize=15, color=INK, pad=28)
    ax.text(0, 1.02, '가입 월별 3개월 유지율 · 2025년',
            transform=ax.transAxes, fontsize=10, color=MUTED,
            ha='left', va='bottom')
    
    # ── R7. 메타 정보 ─────────────────────────────────────────────
    fig.text(0.08, -0.02, SOURCE_NOTE, fontsize=8, color=MUTED, ha='left')

    return fig

# ══════════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    setup_font()

    save(my_answer(), '과제1.png')

#### 문항 2. 이탈이 '언제' 일어나는지 보여 개선 지점을 지목하는 분포 차트 만들기

**상황**

과제 1에서 *"유지율이 17%p 떨어졌다"*는 사실을 보고했습니다. 대표이사가 곧바로 되묻습니다 — "그래서 어디를 손대야 하나?"

이탈이 가입 후 언제 일어나는지 보여 개선 지점을 지목해야 합니다.

청중: 대표이사

청중의 질문: **"어디를 손대야 하는가?"**

차트의 목적: 구성 / 비교 — 어느 구간이 가장 큰가

<br>

**요구사항** 

7개 항목을 모두 만족해야 합니다.

<br> 

**R1.** 차트 종류

가로 막대. 구간명("6~12개월")이 길어 세로 막대는 라벨이 겹치고, 파이는 5조각의 정확한 순위 비교에 약하며 두 구간을 합친 51%를 강조하기 어렵다

**R2.** 순서

정렬하지 않는다. '가입 후 경과 기간'은 순서가 이미 의미를 가지므로 시간 순서를 유지한다

**R3.** 회색조 + 강조 1색

초기 두 구간("1개월 내", "1~3개월")만 색을 주고 나머지는 회색

**R4.** 직접 라벨링

각 막대 끝에 값을 표기하고 x축 눈금은 없앤다

**R5.** 주석

"초기 3개월에 51% 집중" 을 화살표와 함께 강조 구간에 단다

**R6.** Action Title

제목을 결론 문장으로. 부제로 무슨 데이터인지 한 줄

**R7.** 메타 정보

하단에 출처 · 기간 · N수 · 단위 캡션

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ══════════════════════════════════════════════════════════════════════
# 0. 환경 설정 — 수정하지 마세요
# ══════════════════════════════════════════════════════════════════════
INK, BODY, MUTED = '#0c0a09', '#4e4e4e', '#777169'
HAIR, GRAY, GRAY_D = '#d6d3d1', '#c9c5c1', '#a8a29e'
ACCENT, WARN = '#2f6f5e', '#c2543d'

def setup_font():
    '''설치된 한글 폰트를 자동으로 찾아 적용한다.'''
    candidates = [
        'Malgun Gothic',       # Windows
        'AppleGothic',         # macOS
    ]
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            plt.rcParams['font.family'] = name
            break
    else:
        print('[경고] 한글 폰트를 찾지 못했습니다. 라벨이 □로 보일 수 있습니다.\n'
              '       Colab: !apt-get install -y fonts-nanum 후 런타임 재시작')
    plt.rcParams.update({
        'axes.unicode_minus': False,   # 음수 부호 깨짐 방지
    })

def save(fig, path):
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)

# ══════════════════════════════════════════════════════════════════════
# 1. 데이터 — 수정하지 마세요
# ══════════════════════════════════════════════════════════════════════
def load_data():
    '''가입 후 경과 기간별 이탈 비중.
    '''
    return pd.DataFrame({
        '가입후경과': ['1개월 내', '1~3개월', '3~6개월', '6~12개월', '12개월 이상'],
        '이탈비중': [23.0, 28.0, 19.0, 17.0, 13.0],
    })


EARLY = ['1개월 내', '1~3개월']        # 강조할 초기 구간
SOURCE_NOTE = ('자료: 구독 관리 DB · 기간 2025.01~2025.12 · '
               'N=이탈 고객 41,300명 · 단위 %')


# ══════════════════════════════════════════════════════════════════════
# 2. 여기부터 작성하세요
# ══════════════════════════════════════════════════════════════════════
def my_answer():
    df = load_data()

    # 가로 막대는 아래에서 위로 쌓이므로, 시간 순서를 위→아래로 보이게 하려면
    # 데이터를 뒤집어야 합니다. (R2 — 정렬이 아니라 순서 유지)
    d = df.iloc[::-1].reset_index(drop=True)
    y = np.arange(len(d))

    fig, ax = plt.subplots(figsize=(8.6, 4.3))

    # ── R3. 회색조 + 강조 1색 ─────────────────────────────────────
    # TODO: 초기 두 구간(EARLY)만 WARN, 나머지는 GRAY 로 색을 정하세요.
    colors = [WARN if label in EARLY else GRAY for label in d['가입후경과']]

    # ── R1. 가로 막대 ─────────────────────────────────────────────
    bars = ax.barh(y, d['이탈비중'], height=0.62, color=colors, zorder=3)

    # ── R4. 직접 라벨링 ───────────────────────────────────────────
    for yi, val, label in zip(y, d['이탈비중'], d['가입후경과']):
        c = WARN if label in EARLY else GRAY_D
        ax.text(val + 0.8, yi, f'{val:.0f}%', va='center',
                fontsize=10.5, color=c, fontweight='bold')

    # ── R4. x축 눈금 제거 + 정크 정리 ─────────────────────────────
    for side in ['top', 'right', 'bottom', 'left']:
        ax.spines[side].set_visible(False)
    ax.set_xticks([])
    ax.set_xlim(0, 40)

    # ── R5. 주석 ──────────────────────────────────────────────────
    # EARLY 두 구간(뒤집힌 데이터 기준 y=3, y=4)의 중간 지점을 찾는다
    early_ys = [yi for yi, label in zip(y, d['가입후경과']) if label in EARLY]
    mid_y = np.mean(early_ys)

    ax.annotate('초기 3개월에\n51% 집중',
                xy=(30, mid_y), xytext=(37, mid_y - 0.6),
                fontsize=10.5, color=WARN, fontweight='bold',
                ha='center',
                arrowprops=dict(arrowstyle='->', color=WARN, lw=1.3,
                                 connectionstyle='arc3,rad=0.25'))

    # 축 라벨 (참고용)
    ax.set_yticks(y, d['가입후경과'], fontsize=10.5)


    # ── R6. Action Title ──────────────────────────────────────────
    ax.set_title('이탈의 51%가 가입 후 3개월 안에 발생한다', loc='left',
                 fontsize=15, color=INK, pad=28)
    ax.text(0, 1.02, '가입 후 경과 기간별 이탈 비중',
            transform=ax.transAxes, fontsize=10, color=MUTED,
            ha='left', va='bottom')

    # ── R7. 메타 정보 ─────────────────────────────────────────────
    fig.text(0.08, -0.02, SOURCE_NOTE, fontsize=8, color=MUTED, ha='left')

    return fig

if __name__ == '__main__':
    setup_font()

    save(my_answer(), '과제2.png')
